In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

from sklearn.model_selection import KFold, cross_validate

from catboost import CatBoostRegressor


In [2]:
# display all rows
pd.set_option('display.max_rows', None)

##### Download data

In [3]:
# globally set printing options to display all columns
pd.set_option('display.max_columns', None)

# import and visualize data
df = pd.read_csv('data/train.csv')
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


### Set CatBoost baseline-first development

##### Determine categorical features

Given the high dimensionality of this dataset (80 features), I will first implement a baseline CatBoost model. Since CatBoost is optimized to handle categorical features and missing values natively, it allows me to bypass intensive manual preprocessing in this initial phase. My primary goal for this first iteration is to obtain an objective Feature Importance ranking, which will guide my subsequent Exploratory Data Analysis (EDA) and prioritize the features that offer the highest ROI for manual cleaning and engineering.  

Note: While CatBoost handles categories, I must explicitly pass a list of categorical features. I will include numerical features that are 'categorical in disguise' (such as MSSubClass) in this list to prevent the algorithm from incorrectly treating them as continuous variables. 

In [4]:
# split data into X and y
X_catboost = df.drop(columns=['Id', 'SalePrice'])
y = df['SalePrice']
X_catboost.shape, y.shape

((1460, 79), (1460,))

In [5]:
X_catboost.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 79 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   str    
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   str    
 5   Alley          91 non-null     str    
 6   LotShape       1460 non-null   str    
 7   LandContour    1460 non-null   str    
 8   Utilities      1460 non-null   str    
 9   LotConfig      1460 non-null   str    
 10  LandSlope      1460 non-null   str    
 11  Neighborhood   1460 non-null   str    
 12  Condition1     1460 non-null   str    
 13  Condition2     1460 non-null   str    
 14  BldgType       1460 non-null   str    
 15  HouseStyle     1460 non-null   str    
 16  OverallQual    1460 non-null   int64  
 17  OverallCond    1460 non-null   int64  
 18  YearBuilt      1460

In [6]:
# obtain numerical features
num_features = (X_catboost.select_dtypes(include=['int64', 'float64'])
                          .columns
                          .to_list())

After going through all numerical features, the following have been identified as categorical:

`'MSSubClass', 'MoSold'`

In [7]:
# categorical features with a int/float data type
cat_features_as_int = ['MSSubClass', 'MoSold']

# convert data type to str of categorical features previously identified
for feature in cat_features_as_int:
    if feature in num_features:
        X_catboost[feature].astype(str)

# obtain again numerical features after converting some to categorical
num_features = X_catboost.select_dtypes(include=['int64', 'float64']).columns.to_list()

# obtain categorical features
cat_features = X_catboost.select_dtypes(include=['str']).columns.to_list()

# confirm that the total number of features remain the same
len(cat_features), len(num_features)

(43, 36)

In [8]:
# catboost can't handle np.nan in cat features
# fillna with missing value
for feature in cat_features:
    X_catboost[feature] = X_catboost[feature].fillna('missing')


# instantiate the random k fold
cv = KFold(n_splits=5, shuffle=True, random_state=42) # KFold does not care about stratification, stratifiedKfold cannot be used for regression as every label is different

# obtain cross validation scores
cb_reg_score = cross_validate(
    CatBoostRegressor(verbose=0), 
    X_catboost, 
    y, 
    scoring='neg_root_mean_squared_error', 
    cv=cv,
    n_jobs=-1,
    params={'cat_features':cat_features}
    )

# print the baseline cross validated score
print(f"Cross validation score: {cb_reg_score['test_score'].mean()}")

# instantiate and fit a model to obtain feature importance
cb_reg = CatBoostRegressor(cat_features=cat_features, random_seed=42, verbose=0)
cb_reg.fit(X_catboost, y)

cb_feature_importances = pd.DataFrame(cb_reg.feature_importances_,
                                      index=X_catboost.columns, 
                                      columns=['feature_importance']).sort_values(by='feature_importance', ascending=False)
cb_feature_importances.T

Cross validation score: -27152.510281995725


,OverallQual,GrLivArea,GarageCars,TotalBsmtSF,BsmtFinSF1,BsmtQual,ExterQual,1stFlrSF,LotArea,KitchenQual,Neighborhood,FullBath,Fireplaces,BsmtExposure,YearBuilt,2ndFlrSF,OverallCond,GarageFinish,MSZoning,TotRmsAbvGrd,HalfBath,FireplaceQu,SaleCondition,YearRemodAdd,GarageYrBlt,GarageArea,GarageType,OpenPorchSF,MasVnrArea,BsmtFullBath,LandContour,Condition1,LotFrontage,BsmtUnfSF,CentralAir,MoSold,GarageCond,ScreenPorch,BsmtFinType1,WoodDeckSF,BedroomAbvGr,EnclosedPorch,MasVnrType,SaleType,BldgType,Functional,RoofStyle,Foundation,LotConfig,BsmtFinSF2,LotShape,HeatingQC,LandSlope,YrSold,HouseStyle,GarageQual,Fence,Exterior1st,MSSubClass,ExterCond,PoolQC,Exterior2nd,PoolArea,BsmtCond,BsmtFinType2,3SsnPorch,LowQualFinSF,RoofMatl,MiscFeature,Alley,Condition2,PavedDrive,KitchenAbvGr,Electrical,Street,Heating,BsmtHalfBath,Utilities,MiscVal
feature_importance,16.373047,15.475431,6.395476,4.382388,4.307925,3.905277,3.701618,3.635932,3.105042,2.971922,2.79872,2.614579,2.268053,1.839628,1.831756,1.766916,1.501929,1.484625,1.450641,1.385672,1.357243,1.189291,1.138727,1.040902,1.020517,0.947401,0.711879,0.653292,0.564578,0.548435,0.546967,0.416359,0.410081,0.407215,0.401869,0.39633,0.367458,0.300528,0.287166,0.281603,0.271414,0.269632,0.254067,0.232874,0.223531,0.204746,0.197664,0.182732,0.178016,0.174809,0.174102,0.144409,0.131897,0.122417,0.122226,0.113201,0.109529,0.107183,0.08016,0.074583,0.070097,0.06488,0.044192,0.038594,0.033095,0.02656,0.024795,0.021415,0.019353,0.017013,0.016859,0.016427,0.013995,0.0127,0.011699,0.009154,0.005346,0.000208,0.000009
